# MicShift on Colab

Microphone-invariant respiratory audio screening.

**Use this for training and ablations.** Preprocessing (device simulation via
ffmpeg) is CPU-bound and slow everywhere — about 30 s per 25 clips. Training is
what the GPU accelerates: ~40 min on a laptop CPU becomes well under a minute
on a T4.

**Fastest path:** build the feature caches once (locally or here), put the
`.npz` files in Drive, and every future run skips straight to training.

Set **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU'

In [ ]:
!pip install -q librosa soundfile imageio-ffmpeg
import imageio_ffmpeg, torch
print('ffmpeg:', imageio_ffmpeg.get_ffmpeg_exe())
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 1. Get the code

Either clone the repo, or upload the `micshift/` folder plus `config.py`.

In [ ]:
REPO = ""  # e.g. "https://github.com/<you>/micshift.git" - leave blank to upload manually

import os, pathlib
if REPO:
    !git clone -q $REPO micshift_repo && echo cloned
    os.chdir('micshift_repo')
else:
    # Manual: upload a zip of the project (micshift/, config.py) via the Files pane.
    if pathlib.Path('/content/micshift.zip').exists():
        !unzip -qo /content/micshift.zip -d /content/micshift_repo
        os.chdir('/content/micshift_repo')
print('cwd:', os.getcwd())
print('found:', sorted(p.name for p in pathlib.Path('.').glob('*'))[:12])

## 2. Mount Drive and reuse cached features

This is the step that saves the most time. The caches are plain `.npz` holding
already-simulated device features — copy them in and training starts immediately.

Skip to section 3 if you have no caches yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE = '/content/drive/MyDrive/micshift_cache'  # put your .npz files here

import pathlib, shutil
local = pathlib.Path('data/cache'); local.mkdir(parents=True, exist_ok=True)
src = pathlib.Path(DRIVE_CACHE)
if src.exists():
    for f in src.glob('*.npz'):
        shutil.copy(f, local / f.name)
        print('copied', f.name, f'{f.stat().st_size/1e6:.0f} MB')
else:
    print('no Drive cache at', DRIVE_CACHE, '- will build from scratch in section 3')

## 3. Download Coswara (only if you have no cache)

~365 MB per date. `DEFAULT_DATES` is curated for class balance — Coswara's
earliest dates are entirely healthy, and training on them yields a single-class
dataset that reports a meaningless 100% accuracy.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from micshift import data

if not data.local_dates():
    for d in data.DEFAULT_DATES[:8]:
        try:
            data.download_date(d); print('done', d, flush=True)
        except Exception as e:
            print('failed', d, type(e).__name__, flush=True)

clips = data.collect_clips(dates=data.local_dates(), download=False)
import numpy as np
y = np.array([l for _, l, _ in clips])
print(f'{len(clips)} clips, {len({p for _,_,p in clips})} participants, '
      f'{int(y.sum())} positive / {int((y==0).sum())} healthy')

## 4. Build feature caches

CPU-bound (ffmpeg per clip), roughly 1 s per clip per config. Run once, then
save to Drive so you never repeat it.

In [ ]:
from micshift import train as T
dates = data.local_dates()   # pin the corpus so every config sees identical data

for invert in (False, True):
    print(f'--- cache invert={invert} ---', flush=True)
    T.build_cache(n_devices=3, invert=invert, dates=dates)

# Persist so future sessions skip all of the above.
import shutil, pathlib
out = pathlib.Path(DRIVE_CACHE); out.mkdir(parents=True, exist_ok=True)
for f in pathlib.Path('data/cache').glob('*.npz'):
    shutil.copy(f, out / f.name)
print('saved caches to', out)

## 5. Train (GPU)

`train()` picks up CUDA automatically.

In [ ]:
import time, torch
from micshift import model as M, evaluate as E, channel

cached = T.build_cache(n_devices=3, invert=True, dates=dates)
t0 = time.time()
net, val_bal = T.train(cached, epochs=60, verbose=True)
print(f'trained in {time.time()-t0:.0f}s   val balanced acc = {val_bal:.3f}')

Xte, yte, mte, _ = cached['test']
Xm,  ym,  _,  _  = cached['test_matched']
_, unseen = E.metrics(M.predict(net, Xte, device='cuda' if torch.cuda.is_available() else 'cpu'), yte)
_, matched = E.metrics(M.predict(net, Xm,  device='cuda' if torch.cuda.is_available() else 'cpu'), ym)
print(f'matched={matched:.3f}  unseen={unseen:.3f}  retention={unseen/matched:.1%}')

## 6. Full ablation

Compares MicShift against no-inversion and SpecAugment. Results are written
per-config to `data/results.json` as each finishes, so an interrupted run keeps
completed work.

In [ ]:
!python -W ignore -u -m micshift.evaluate --devices 3 --epochs 60

In [ ]:
import json, pandas as pd
rows = json.loads(open('data/results.json').read())
cols = ['name','matched_balanced_acc','unseen_balanced_acc','retention',
        'abstain_rate','kept_acc','precision','base_error']
pd.DataFrame(rows)[cols].round(3)

## 7. Save the model back to Drive

In [ ]:
import torch, json, pathlib, shutil
from micshift import channel

torch.save(net.state_dict(), 'data/model.pt')
bound = channel.calibrate_bound(cached['val'][2])
pathlib.Path('data/bound.json').write_text(json.dumps({'bound': bound}))
print(f'calibrated abstention bound: {bound:.2f} dB')

dest = pathlib.Path('/content/drive/MyDrive/micshift_out'); dest.mkdir(parents=True, exist_ok=True)
for f in ('data/model.pt', 'data/bound.json', 'data/results.json'):
    if pathlib.Path(f).exists():
        shutil.copy(f, dest / pathlib.Path(f).name)
print('saved to', dest)